In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

In [2]:
def select_features(donors_df, pairs_df, DATA_DIR, chosen_region, target = "6e10",
                    rho_cutoff = 0.2, pval_cutoff = 0.001, if_stringent = False, N_ROUNDS = 10):
    features_round = []

    for r in range(N_ROUNDS):

        print(f"\n----------- Processing round {r} -----------\n")

        val_donors = donors_df[donors_df[f"val_r{r}"] == 1].index

        results_assoc = pd.read_csv(os.path.join(DATA_DIR, f"{chosen_region}_split_assoc_{target}/pairs_LW_assoc_r{r}.csv"))

        results_selected = results_assoc[(abs(results_assoc["rho"]) > rho_cutoff) & (results_assoc["pval"] < pval_cutoff)]
        selected_features = results_selected['feature'].values
        print(f" Correlation based selected features (in discovery set): {len(selected_features)}")

        val_df = pairs_df.loc[val_donors, selected_features]
        target_col = f"percent {target} positive area"
        y_val = donors_df.loc[val_donors, target_col]

        rho_val = [spearmanr(val_df[f], y_val)[0] for f in selected_features]
        results_selected = results_selected.copy()
        results_selected["rho_val"] = rho_val
        
        if if_stringent:
            stable = results_selected[((results_selected["rho"] * results_selected["rho_val"]) > 0) &
                                    (abs(results_selected["rho_val"]) > rho_cutoff)]  # same sign + rho_val > rho_cutoff
            stable_features = stable['feature'].values
            print(f" Stable stable features (in replication set): {len(stable_features)}")
        else:
            stable = results_selected[(results_selected["rho"] * results_selected["rho_val"]) > 0]  # same sign
            stable_features = stable['feature'].values
            print(f"  Stable features (in replication set): {len(stable_features)}")

        features_round.append(stable_features)

    return features_round



# SELECTED IN MTG (Microglia-PVM)

In [ ]:
chosen_region = 'MTG'
donors_df = pd.read_csv(f"data/{chosen_region}_donors_split.csv", index_col=0)

In [ ]:
chosen_cell_type = 'Microglia-PVM'  
celltype_sanitized = re.sub(r'(\d+)/(\d+)', r'\1-\2', chosen_cell_type)  
celltype_sanitized = re.sub(r'\s+', '_', celltype_sanitized)  

DATA_DIR = f"results/pair_features/{celltype_sanitized}/"

with open(os.path.join(DATA_DIR, f"{chosen_region}_pair_features_LW_median.pkl"), "rb") as f:
     pairs_median_df = pickle.load(f)

## 6e10 

In [5]:
features_round = select_features(donors_df, pairs_median_df, DATA_DIR, chosen_region, target = "6e10",
                                 rho_cutoff = 0.2, pval_cutoff = 0.001, if_stringent = True, N_ROUNDS = 10)


----------- Processing round 0 -----------

 Correlation based selected features (in discovery set): 312
 Stable stable features (in replication set): 51

----------- Processing round 1 -----------

 Correlation based selected features (in discovery set): 579
 Stable stable features (in replication set): 72

----------- Processing round 2 -----------

 Correlation based selected features (in discovery set): 431
 Stable stable features (in replication set): 54

----------- Processing round 3 -----------

 Correlation based selected features (in discovery set): 426
 Stable stable features (in replication set): 58

----------- Processing round 4 -----------

 Correlation based selected features (in discovery set): 1740
 Stable stable features (in replication set): 190

----------- Processing round 5 -----------

 Correlation based selected features (in discovery set): 479
 Stable stable features (in replication set): 70

----------- Processing round 6 -----------

 Correlation based sele

In [6]:
all_features = [f for sublist in features_round for f in sublist]
freq = Counter(all_features)
freq_df = pd.DataFrame(freq.items(), columns=["feature", "count"]).sort_values("count", ascending=False)
print(freq_df[freq_df['count'] >= 5])

min_rounds = 5
selected_features = [f for f, c in freq.items() if c >= min_rounds]
print(f"\nFinal selected features (appeared in at least {min_rounds} rounds): \n{selected_features}")

              feature  count
29         PDE4B_ZEB1      7
79       IRS2_ST3GAL6      6
97         IPCEF1_CPM      6
6    SERPINB9_ATXN7L1      5
76          OXR1_RGL1      5
65         PDGFB_LCP2      5
143        MSR1_PTPRG      5
68          EVL_ARMH3      5
63    GRID2_LINC01684      5

Final selected features (appeared in at least 5 rounds): 
['SERPINB9_ATXN7L1', 'PDE4B_ZEB1', 'GRID2_LINC01684', 'PDGFB_LCP2', 'EVL_ARMH3', 'OXR1_RGL1', 'IRS2_ST3GAL6', 'IPCEF1_CPM', 'MSR1_PTPRG']


## AT8

In [7]:
features_round = select_features(donors_df, pairs_median_df, DATA_DIR, chosen_region, target = "AT8",
                                 rho_cutoff = 0.2, pval_cutoff = 0.001, if_stringent = True, N_ROUNDS = 10)


----------- Processing round 0 -----------

 Correlation based selected features (in discovery set): 1163
 Stable stable features (in replication set): 272

----------- Processing round 1 -----------

 Correlation based selected features (in discovery set): 578
 Stable stable features (in replication set): 151

----------- Processing round 2 -----------

 Correlation based selected features (in discovery set): 980
 Stable stable features (in replication set): 267

----------- Processing round 3 -----------

 Correlation based selected features (in discovery set): 669
 Stable stable features (in replication set): 168

----------- Processing round 4 -----------

 Correlation based selected features (in discovery set): 1017
 Stable stable features (in replication set): 276

----------- Processing round 5 -----------

 Correlation based selected features (in discovery set): 880
 Stable stable features (in replication set): 258

----------- Processing round 6 -----------

 Correlation base

In [8]:
all_features = [f for sublist in features_round for f in sublist]
freq = Counter(all_features)
freq_df = pd.DataFrame(freq.items(), columns=["feature", "count"]).sort_values("count", ascending=False)
print(freq_df[freq_df['count'] >= 7])

min_rounds = 7
selected_features = [f for f, c in freq.items() if c >= min_rounds]
print(f"\nFinal selected features (appeared in at least {min_rounds} rounds): \n{selected_features}")


            feature  count
29       GAS7_LRMDA      8
284     PTPRG_ANOS1      8
100   STARD13_FOXP1      8
1        DLEU1_SNCA      7
32      FMNL2_PTPRG      7
273    MT-CO2_PLCG2      7
329      MSR1_PTPRG      7
94      ACSL1_PTPRG      7
257  SLC11A1_MVB12B      7

Final selected features (appeared in at least 7 rounds): 
['DLEU1_SNCA', 'GAS7_LRMDA', 'FMNL2_PTPRG', 'ACSL1_PTPRG', 'STARD13_FOXP1', 'SLC11A1_MVB12B', 'MT-CO2_PLCG2', 'PTPRG_ANOS1', 'MSR1_PTPRG']


# SELECTED IN A9 (Microglia-PVM)

In [ ]:
chosen_region = 'A9'
donors_df = pd.read_csv(f"data/{chosen_region}_donors_split.csv", index_col=0)

In [ ]:
chosen_cell_type = 'Microglia-PVM'  
celltype_sanitized = re.sub(r'(\d+)/(\d+)', r'\1-\2', chosen_cell_type)  
celltype_sanitized = re.sub(r'\s+', '_', celltype_sanitized)  

DATA_DIR = f"results/pair_features/{celltype_sanitized}/"

with open(os.path.join(DATA_DIR, f"{chosen_region}_pair_features_LW_median.pkl"), "rb") as f:
     pairs_median_df = pickle.load(f)

## 6e10

In [11]:
features_round = select_features(donors_df, pairs_median_df, DATA_DIR, chosen_region, target = "6e10",
                                rho_cutoff = 0.2, pval_cutoff = 0.001, if_stringent = True, N_ROUNDS = 10)


----------- Processing round 0 -----------

 Correlation based selected features (in discovery set): 275
 Stable stable features (in replication set): 38

----------- Processing round 1 -----------

 Correlation based selected features (in discovery set): 888
 Stable stable features (in replication set): 64

----------- Processing round 2 -----------

 Correlation based selected features (in discovery set): 216
 Stable stable features (in replication set): 38

----------- Processing round 3 -----------

 Correlation based selected features (in discovery set): 562
 Stable stable features (in replication set): 58

----------- Processing round 4 -----------

 Correlation based selected features (in discovery set): 1361
 Stable stable features (in replication set): 67

----------- Processing round 5 -----------

 Correlation based selected features (in discovery set): 275
 Stable stable features (in replication set): 32

----------- Processing round 6 -----------

 Correlation based selec

In [12]:
all_features = [f for sublist in features_round for f in sublist]
freq = Counter(all_features)
freq_df = pd.DataFrame(freq.items(), columns=["feature", "count"]).sort_values("count", ascending=False)
print(freq_df[freq_df['count'] >= 5])

min_rounds = 5
selected_features = [f for f, c in freq.items() if c >= min_rounds]
print(f"\nFinal selected features (appeared in at least {min_rounds} rounds): \n{selected_features}")

              feature  count
16     PTPRG_RALGAPA2      6
43          PTPRG_CPM      6
40        ERBIN_APPL2      5
45  ST6GALNAC3_OSBPL8      5
46        MSRA_SMURF2      5

Final selected features (appeared in at least 5 rounds): 
['PTPRG_RALGAPA2', 'ERBIN_APPL2', 'PTPRG_CPM', 'ST6GALNAC3_OSBPL8', 'MSRA_SMURF2']


## AT8

In [13]:
features_round = select_features(donors_df, pairs_median_df, DATA_DIR, chosen_region, target = "AT8",
                                rho_cutoff = 0.2, pval_cutoff = 0.001, if_stringent = True, N_ROUNDS = 10)


----------- Processing round 0 -----------

 Correlation based selected features (in discovery set): 3036
 Stable stable features (in replication set): 843

----------- Processing round 1 -----------

 Correlation based selected features (in discovery set): 2125
 Stable stable features (in replication set): 516

----------- Processing round 2 -----------

 Correlation based selected features (in discovery set): 1967
 Stable stable features (in replication set): 526

----------- Processing round 3 -----------

 Correlation based selected features (in discovery set): 1002
 Stable stable features (in replication set): 280

----------- Processing round 4 -----------

 Correlation based selected features (in discovery set): 3711
 Stable stable features (in replication set): 948

----------- Processing round 5 -----------

 Correlation based selected features (in discovery set): 6496
 Stable stable features (in replication set): 1011

----------- Processing round 6 -----------

 Correlation

In [14]:
all_features = [f for sublist in features_round for f in sublist]
freq = Counter(all_features)
freq_df = pd.DataFrame(freq.items(), columns=["feature", "count"]).sort_values("count", ascending=False)
print(freq_df[freq_df['count'] >= 8])

min_rounds = 8
selected_features = [f for f, c in freq.items() if c >= min_rounds]
print(f"\nFinal selected features (appeared in at least {min_rounds} rounds): \n{selected_features}")

             feature  count
336    MT-ND2_SPRED1      9
99   MARCH3_ARHGAP31      9
483     ADGRB3_MARK3      8
50        RB1_IL1RAP      8
1       PTPRG_SAMD4A      8
219    CACNA1D_ITPR1      8
8         PTPRG_MITF      8
147       PTPRG_MSR1      8
92       PTPRG_MYO1E      8

Final selected features (appeared in at least 8 rounds): 
['PTPRG_SAMD4A', 'PTPRG_MITF', 'RB1_IL1RAP', 'PTPRG_MYO1E', 'MARCH3_ARHGAP31', 'PTPRG_MSR1', 'CACNA1D_ITPR1', 'MT-ND2_SPRED1', 'ADGRB3_MARK3']
